# OCCLUDE on Colab (v0.1 — offline, three-pass)

Blur immodestly dressed people in a video, audio intact.

**Before you run:** set a GPU runtime — `Runtime ▸ Change runtime type ▸ GPU` (A100/L4/T4 all work; bigger is faster). OCCLUDE runs a high-recall detector (RT-DETR), SAM2, and a 7B vision-language model (Qwen2.5-VL), so CPU is not practical for a full video.

**Pipeline:** Pass 1 detect + track people into per-person tracklets → Pass 2 a VLM judges each person once from their clearest frames → Pass 3 render the verdict across their whole on-screen span and mux the audio back.

If you also want background music removed, run ELUATE on the file first, then feed its output here.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install OCCLUDE, SAM2, and ffmpeg

SAM2 isn't on PyPI, so it installs from source (it gives the clean silhouette-shaped blur; without it, pass `--no-sam2` for a feathered-box blur).

In [ ]:
!pip -q install occlude
!pip -q install "git+https://github.com/facebookresearch/sam2.git"
!apt-get -qq install -y ffmpeg > /dev/null
import occlude; print('occlude', occlude.__version__)

## 3. Get a video in

Run **one** of the two cells: upload from your machine, or mount Drive and point at a path.

In [ ]:
# Option A — upload from your computer
from google.colab import files
uploaded = files.upload()
SRC = next(iter(uploaded))
print('source:', SRC)

In [ ]:
# Option B — from Google Drive (edit the path)
# from google.colab import drive
# drive.mount('/content/drive')
# SRC = '/content/drive/MyDrive/videos/documentary.mp4'
# print('source:', SRC)

## 4. Transcode to H.264

OCCLUDE decodes with OpenCV, which can't read **AV1** (common for Google/YouTube-sourced files). One ffmpeg pass to H.264 bridges that. It's quick and harmless if the file is already H.264, so just run it.

In [ ]:
import subprocess
INPUT = '/content/input_h264.mp4'
subprocess.run([
    'ffmpeg', '-y', '-hide_banner', '-loglevel', 'error', '-i', SRC,
    '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '20', '-c:a', 'copy', INPUT,
], check=True)
print('ready:', INPUT)

## 5. Run OCCLUDE

First run downloads the model weights (RT-DETR, SAM2, Qwen2.5-VL ~7B) — expect a few minutes before the progress bars move. A feature-length video then takes a few hours on one GPU; **try a short clip first.**

Optional knobs: `--judge-batch 16` (throughput), `--judge-frames 5` (accuracy), `--blur-strength 199` (blur radius, odd), `--no-sam2` (box blur if the SAM2 install failed).

In [ ]:
OUTPUT = '/content/output_occluded.mp4'
!occlude --input "$INPUT" --output "$OUTPUT" --device cuda
print('output:', OUTPUT)

## 6. Download the result

In [ ]:
from google.colab import files
files.download(OUTPUT)